# release16_action_head_position_loss_screen_v1

Purpose: continue the handoff from `release16_action_head_position_loss_screen_v1` by testing whether the failed BC action head is a pure cross-entropy calibration problem. This screen trains position/ordinal-aware BC actors on the same aux state-head joint target contract, audits deterministic holdout sparse slices, and replays each actor through official qlib warm-start walk-forward windows.

Baselines:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison baseline: `release16_action_head_position_loss_screen_v1`.
- Target-contract source: `release16_joint_drawdown_release_target_contract_v1`.
- Performance reference: `soxlbacktest_qlib_walkforward_baseline_v1`.

Role: diagnose.

Changed layer: `model_selection`, `bc_warm_start`, `bc_position_loss`, `official_replay`.

Target window: `wf_2020`, `wf_2022`, `wf_2023_2024`; no actor can be promoted or retained from this screen.

Hypothesis: the previous BC screen failed because a pure cross-entropy action head treats adjacent position-label errors and large de-risk/re-entry errors too similarly. Adding an ordinal action-distance penalty should reduce holdout action-distance error and improve downtrend/rebound slice generalization before any PPO or retention step.

Acceptance criteria:
- The same fold-local joint target contract is used with deterministic non-overlapping train/holdout target rows.
- A selected BC config passes conditional holdout sparse-slice gates in every fold.
- Official warm-start replay passes `relative_return >= 30pp`, `MaxDD < 20%`, and drift checks versus the runnable reference in every fold.
- Action calibration sidecars show holdout action-distance errors for all required slices.
- SOXLBacktest qlib baseline comparison is written as sidecar evidence.
- Summary explicitly keeps `promotion_allowed=false`, `actor_retention_allowed=false`, and `ppo_allowed_next=false`.

Closest success: `release16_joint_drawdown_release_target_contract_v1` produced a fold-local joint target contract ready for BC screening.

Closest failure: `release16_action_head_position_loss_screen_v1` had train slice fit but failed holdout slice gates and official replay gates in all three walk-forward windows.

Why this is not a duplicate: it keeps the same target contract and replay gate, but changes the action-head loss from CE-only to CE plus ordinal action-distance penalty, and adds explicit holdout action-calibration sidecars.


In [1]:

from __future__ import annotations

from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import math
import os
import random
import sys

import numpy as np
import pandas as pd
import torch

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics, max_drawdown_interval_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCFitResult,
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    action_distribution_frame,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor,
    evaluate_bc_actor_slice_fit,
    score_bc_walkforward_drift,
    state_objective_bc_sample_weights,
    summarize_bc_walkforward_drift_screen,
)
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import build_warm_start_replay_segment, slice_warm_start_daily_path
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'release16_action_head_position_loss_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / 'official_daily_paths'
DAILY_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CONTRACT_CARD = ROOT / 'runtime/model_selection/card/release16_joint_drawdown_release_target_contract_v1'
PRIOR_REPLAY_CARD = ROOT / 'runtime/model_selection/card/release16_nonconvex_checkpoint_bc_official_replay_v1'
SOURCE_SCORE_CARD = ROOT / 'runtime/model_selection/card/release16_nonconvex_checkpoint_objective_score_handoff_v1'
SOXL_BASELINE_CARD = ROOT / 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1'
TARGETS_PATH = SOURCE_CONTRACT_CARD / 'joint_target_contract.csv'
SOURCE_SUMMARY_PATH = SOURCE_CONTRACT_CARD / 'summary.json'
RUNNABLE_REFERENCE_PATH = SOURCE_SCORE_CARD / 'best_runnable_window_terms.csv'
SOXL_BASELINE_METRICS_PATH = SOXL_BASELINE_CARD / 'candidate_metrics.csv'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260615
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0
DRIFT_RELATIVE_FLOOR_PP = -5.0
DRIFT_MAX_DD_CEILING_PP = 2.0
HOLDOUT_MODULUS = 5
AFFECTED_FOLDS = {'wf_2022', 'wf_2023_2024'}

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_action_head_position_loss_screen_v1


In [2]:

WALK_FORWARD_WINDOWS = {
    'wf_2020': {'target_fold': 'wf_2020_train', 'train': ('2010-03-11', '2019-12-31'), 'valid': ('2020-01-01', '2020-12-31'), 'test': ('2020-01-01', '2020-12-31')},
    'wf_2022': {'target_fold': 'wf_2022_train', 'train': ('2010-03-11', '2020-12-31'), 'valid': ('2021-01-01', '2021-12-31'), 'test': ('2022-01-01', '2022-12-31')},
    'wf_2023_2024': {'target_fold': 'wf_2023_2024_train', 'train': ('2010-03-11', '2022-12-31'), 'valid': ('2023-01-01', '2023-12-31'), 'test': ('2024-01-01', '2024-12-31')},
}
BC_TRAIN_CONFIGS = {
    'ordinal_low_aux': StateObjectiveBCTrainingConfig(steps=360, lr=0.0008, batch_size=128, seed=SEED + 101),
    'ordinal_mid_aux': StateObjectiveBCTrainingConfig(steps=420, lr=0.0007, batch_size=128, seed=SEED + 137),
    'ordinal_high_aux': StateObjectiveBCTrainingConfig(steps=460, lr=0.0006, batch_size=128, seed=SEED + 173),
    'ordinal_drawdown_aux': StateObjectiveBCTrainingConfig(steps=460, lr=0.0006, batch_size=128, seed=SEED + 211),
}
BC_WEIGHT_CONFIGS = {
    'ordinal_low_aux': StateObjectiveBCSampleWeightConfig(rebound_multiplier=5.0, downtrend_multiplier=6.0, release_gate_multiplier=16.0, ordinary_multiplier=1.0, objective_weight_power=0.45, class_balanced_beta=0.94, weight_floor=0.05, weight_cap=55.0, normalize_mean=True),
    'ordinal_mid_aux': StateObjectiveBCSampleWeightConfig(rebound_multiplier=6.0, downtrend_multiplier=8.0, release_gate_multiplier=18.0, ordinary_multiplier=1.0, objective_weight_power=0.50, class_balanced_beta=0.96, weight_floor=0.05, weight_cap=65.0, normalize_mean=True),
    'ordinal_high_aux': StateObjectiveBCSampleWeightConfig(rebound_multiplier=6.0, downtrend_multiplier=10.0, release_gate_multiplier=18.0, ordinary_multiplier=1.0, objective_weight_power=0.55, class_balanced_beta=0.97, weight_floor=0.05, weight_cap=70.0, normalize_mean=True),
    'ordinal_drawdown_aux': StateObjectiveBCSampleWeightConfig(rebound_multiplier=4.0, downtrend_multiplier=16.0, release_gate_multiplier=12.0, ordinary_multiplier=1.0, objective_weight_power=0.55, class_balanced_beta=0.96, weight_floor=0.05, weight_cap=75.0, normalize_mean=True),
}
POSITION_LOSS_CONFIGS = {
    'ordinal_low_aux': {'ordinal_lambda': 0.40},
    'ordinal_mid_aux': {'ordinal_lambda': 1.25},
    'ordinal_high_aux': {'ordinal_lambda': 2.50},
    'ordinal_drawdown_aux': {'ordinal_lambda': 1.75},
}

# Full coverage is audited separately; holdout gates use lower sample minimums because the split is 20% of the train target rows.
SLICE_RULES = {
    'all': {'min_samples': 50, 'min_accuracy': 0.50, 'min_predicted_unique': 2, 'conditional': 'always'},
    'aux_state_head_release_state': {'min_samples': 5, 'min_accuracy': 0.60, 'min_predicted_unique': 2, 'conditional': 'affected_if_available'},
    'is_rebound_cluster': {'min_samples': 5, 'min_accuracy': 0.60, 'min_predicted_unique': 2, 'conditional': 'if_available'},
    'is_downtrend_state': {'min_samples': 5, 'min_accuracy': 0.60, 'min_predicted_unique': 2, 'conditional': 'if_available'},
    'drawdown_control_state': {'min_samples': 3, 'min_accuracy': 0.60, 'min_predicted_unique': 1, 'conditional': 'if_available'},
    'failedfold_state_head_target_state': {'min_samples': 5, 'min_accuracy': 0.55, 'min_predicted_unique': 2, 'conditional': 'if_available'},
    # Original release_gate_state is retained as a diagnostic because the source root cause was sparsity;
    # aux_state_head_release_state is the hard release-repair gate for affected folds.
    'release_gate_state': {'min_samples': 3, 'min_accuracy': 0.55, 'min_predicted_unique': 1, 'conditional': 'diagnostic_only'},
}
SLICE_COLS = tuple(name for name in SLICE_RULES if name != 'all')
PROTECTED_HOLDOUT_SLICES = {
    'aux_state_head_release_state': 5,
    'is_rebound_cluster': 5,
    'is_downtrend_state': 5,
    'drawdown_control_state': 5,
    'failedfold_state_head_target_state': 5,
    'release_gate_state': 3,
}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return str(value.date())
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def bool_series(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(False, index=frame.index)
    return frame[column].fillna(False).map(lambda value: bool(value) if isinstance(value, bool) else str(value).strip().lower() in {'1', 'true', 'yes', 'y'})


def datetime_index_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if isinstance(out.index, pd.MultiIndex):
        if 'instrument' in out.index.names:
            out = out.droplevel('instrument')
        else:
            out = out.reset_index(level=list(range(1, out.index.nlevels)), drop=True)
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index))
    return out[~out.index.duplicated(keep='last')].sort_index()


def datetime_index_series(series: pd.Series) -> pd.Series:
    out = series.copy()
    if isinstance(out.index, pd.MultiIndex):
        if 'instrument' in out.index.names:
            out = out.droplevel('instrument')
        else:
            out.index = out.index.get_level_values(0)
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index))
    return out[~out.index.duplicated(keep='last')].sort_index()


def baseline_metrics_by_fold(frame: pd.DataFrame, candidate: str | None = None) -> dict[str, dict[str, float]]:
    out: dict[str, dict[str, float]] = {}
    work = frame.copy()
    if candidate and 'candidate' in work.columns:
        work = work[work['candidate'].astype(str).eq(candidate)]
    for _, row in work.iterrows():
        fold = str(row.get('window') or row.get('fold') or '')
        if not fold.startswith('wf_'):
            continue
        rel = row.get('relative_return')
        if (rel is None or pd.isna(rel)) and {'rulebase_return_pct', 'soxl_buyhold_return_pct'}.issubset(work.columns):
            rel = float(row['rulebase_return_pct']) - float(row['soxl_buyhold_return_pct'])
        max_dd = row.get('max_dd')
        if (max_dd is None or pd.isna(max_dd)) and 'rulebase_max_drawdown_pct' in work.columns:
            max_dd = row['rulebase_max_drawdown_pct']
        avg_pos = row.get('avg_pos')
        if (avg_pos is None or pd.isna(avg_pos)) and 'average_position_pct' in work.columns:
            avg_pos = float(row['average_position_pct']) / 100.0
        out[fold] = {
            'relative_return': float(rel),
            'max_dd': float(max_dd),
            'avg_pos': float(avg_pos),
        }
    return out


In [3]:

def stratified_target_split(targets: pd.DataFrame) -> pd.DataFrame:
    work = targets.sort_values('datetime').reset_index(drop=True).copy()
    work['_target_row_number'] = np.arange(len(work), dtype=int)
    holdout_idx = set(work.index[work['_target_row_number'] % HOLDOUT_MODULUS == HOLDOUT_MODULUS - 1].tolist())
    for column, min_holdout in PROTECTED_HOLDOUT_SLICES.items():
        if column not in work.columns:
            continue
        idxs = work.index[bool_series(work, column)].tolist()
        if len(idxs) < int(min_holdout) + 1:
            continue
        desired = min(max(int(min_holdout), int(math.ceil(len(idxs) * 0.20))), len(idxs) - 1)
        current = [idx for idx in idxs if idx in holdout_idx]
        if len(current) >= desired:
            continue
        for idx in reversed(idxs):
            if idx in holdout_idx:
                continue
            holdout_idx.add(idx)
            current.append(idx)
            if len(current) >= desired:
                break
    work['bc_target_split'] = np.where(work.index.isin(holdout_idx), 'holdout', 'train')
    return work


def slice_gate_from_metrics(
    slice_metrics: pd.DataFrame,
    *,
    fold: str,
    full_target_counts: dict[str, int],
) -> tuple[bool, dict[str, Any], list[str]]:
    details: dict[str, Any] = {}
    failed: list[str] = []
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    for name, rule in SLICE_RULES.items():
        conditional = str(rule['conditional'])
        full_count = int(full_target_counts.get(name, 0 if name != 'all' else full_target_counts.get('all', 0)))
        required = conditional == 'always'
        if conditional == 'diagnostic_only':
            required = False
        elif conditional == 'if_available':
            required = full_count >= int(rule['min_samples'])
        elif conditional == 'affected_if_available':
            required = fold in AFFECTED_FOLDS and full_count >= int(rule['min_samples'])
        if not required:
            details[name] = {
                'required': False,
                'full_target_count': full_count,
                'samples': 0,
                'accuracy': None,
                'gate': True,
                'skip_reason': 'not_required_or_insufficient_full_slice_rows',
            }
            continue
        if name not in indexed.index:
            details[name] = {
                'required': True,
                'full_target_count': full_count,
                'samples': 0,
                'accuracy': None,
                'gate': False,
                'failed_reason': 'missing_slice_metric',
            }
            failed.append(name)
            continue
        row = indexed.loc[name]
        samples = int(row.get('samples', 0))
        accuracy = float(row.get('accuracy', float('nan')))
        label_unique = int(row.get('label_unique_actions', 0))
        predicted_unique = int(row.get('predicted_unique_actions', 0))
        min_predicted_unique = int(rule.get('min_predicted_unique', 1)) if label_unique >= 2 else 1
        sample_ok = samples >= int(rule['min_samples'])
        accuracy_ok = math.isfinite(accuracy) and accuracy >= float(rule['min_accuracy'])
        diversity_ok = predicted_unique >= min_predicted_unique
        ok = sample_ok and accuracy_ok and diversity_ok
        details[name] = {
            'required': True,
            'full_target_count': full_count,
            'samples': samples,
            'accuracy': accuracy,
            'label_unique_actions': label_unique,
            'predicted_unique_actions': predicted_unique,
            'min_samples': int(rule['min_samples']),
            'min_accuracy': float(rule['min_accuracy']),
            'min_predicted_unique': min_predicted_unique,
            'sample_gate': bool(sample_ok),
            'accuracy_gate': bool(accuracy_ok),
            'diversity_gate': bool(diversity_ok),
            'gate': bool(ok),
        }
        if not ok:
            failed.append(name)
    return (len(failed) == 0), details, failed


def candidate_loss(row: dict[str, Any]) -> float:
    relative_gap = max(0.0, TARGET_RELATIVE_RETURN - float(row.get('relative_return', float('-inf'))))
    dd_gap = max(0.0, float(row.get('max_dd', float('inf'))) - TARGET_MAX_DD)
    holdout_penalty = 25.0 * int(not bool(row.get('holdout_slice_gate')))
    train_penalty = 5.0 * int(not bool(row.get('train_slice_gate')))
    drift_penalty = max(0.0, DRIFT_RELATIVE_FLOOR_PP - float(row.get('relative_delta_vs_runnable', float('-inf'))))
    drift_penalty += max(0.0, float(row.get('max_dd_delta_vs_runnable', float('inf'))) - DRIFT_MAX_DD_CEILING_PP) * 2.0
    action_collapse_penalty = max(0, 2 - int(row.get('holdout_required_predicted_unique_actions_min', 0))) * 10.0
    holdout_action_mae = float(row.get('holdout_action_mae_all', float('inf')))
    severe_error_share = float(row.get('holdout_severe_action_error_share_all', 1.0))
    action_distance_penalty = max(0.0, holdout_action_mae - 0.80) * 8.0 + max(0.0, severe_error_share - 0.25) * 10.0
    return float(relative_gap + dd_gap * 4.0 + holdout_penalty + train_penalty + drift_penalty + action_collapse_penalty + action_distance_penalty)

def slice_count_map(targets: pd.DataFrame) -> dict[str, int]:
    counts = {'all': int(len(targets))}
    for col in SLICE_COLS:
        counts[col] = int(bool_series(targets, col).sum()) if col in targets.columns else 0
    return counts


In [4]:

def _actor_logits_for_notebook(actor: Any, obs_tensor: torch.Tensor) -> torch.Tensor:
    out = actor(obs_tensor)
    return out[0] if isinstance(out, tuple) else out


def _actor_predictions_for_notebook(actor: Any, obs_array: np.ndarray, *, batch_size: int = 4096) -> np.ndarray:
    if len(obs_array) == 0:
        return np.asarray([], dtype=np.int64)
    device = next(actor.parameters()).device
    obs_tensor = torch.as_tensor(obs_array, dtype=torch.float32, device=device)
    preds: list[np.ndarray] = []
    with torch.no_grad():
        for start in range(0, int(obs_tensor.shape[0]), int(batch_size)):
            stop = min(start + int(batch_size), int(obs_tensor.shape[0]))
            logits = _actor_logits_for_notebook(actor, obs_tensor[start:stop])
            preds.append(logits.argmax(-1).detach().cpu().numpy().astype(np.int64))
    return np.concatenate(preds)


def _position_aware_supervised_train(
    actor: Any,
    obs_array: np.ndarray,
    action_array: np.ndarray,
    sample_weight_array: np.ndarray,
    *,
    steps: int,
    lr: float,
    batch_size: int,
    ordinal_lambda: float,
) -> dict[str, float | int]:
    if int(steps) <= 0 or len(obs_array) == 0:
        return {"loss": float("nan"), "accuracy": float("nan"), "ordinal_loss": float("nan"), "ce_loss": float("nan"), "steps": 0}
    device = next(actor.parameters()).device
    obs_tensor = torch.as_tensor(obs_array, dtype=torch.float32, device=device)
    action_tensor = torch.as_tensor(action_array, dtype=torch.long, device=device)
    weight_tensor = torch.as_tensor(sample_weight_array, dtype=torch.float32, device=device)
    n = int(obs_tensor.shape[0])
    optim = torch.optim.Adam(actor.parameters(), lr=float(lr))
    ce_fn = torch.nn.CrossEntropyLoss(reduction='none')
    last_loss = last_ce = last_ord = last_acc = float('nan')
    for _ in range(int(steps)):
        idx = torch.randint(0, n, (min(int(batch_size), n),), device=device)
        logits = _actor_logits_for_notebook(actor, obs_tensor[idx])
        labels = action_tensor[idx]
        weights = weight_tensor[idx]
        ce = ce_fn(logits, labels)
        action_axis = torch.arange(logits.shape[-1], dtype=torch.float32, device=device)
        denom = max(float(logits.shape[-1] - 1), 1.0)
        probs = torch.softmax(logits, dim=-1)
        expected_action = (probs * action_axis).sum(dim=-1) / denom
        target_action = labels.float() / denom
        ordinal = torch.square(expected_action - target_action)
        weighted_ce = (ce * weights).sum() / weights.sum().clamp_min(1e-8)
        weighted_ordinal = (ordinal * weights).sum() / weights.sum().clamp_min(1e-8)
        loss = weighted_ce + float(ordinal_lambda) * weighted_ordinal
        optim.zero_grad()
        loss.backward()
        optim.step()
        last_loss = float(loss.detach().cpu().item())
        last_ce = float(weighted_ce.detach().cpu().item())
        last_ord = float(weighted_ordinal.detach().cpu().item())
        with torch.no_grad():
            preds = logits.argmax(-1)
            last_acc = float((preds == labels).float().mean().item())
    return {"loss": last_loss, "ce_loss": last_ce, "ordinal_loss": last_ord, "accuracy": last_acc, "steps": int(steps)}


def fit_position_aware_bc_actor_from_frame(
    df: pd.DataFrame,
    *,
    targets: pd.DataFrame,
    profile: Any,
    train_config: StateObjectiveBCTrainingConfig,
    sample_weight_config: StateObjectiveBCSampleWeightConfig,
    position_loss_config: dict[str, float],
    sample_size: int = 0,
    rng_seed: int = 0,
) -> StateObjectiveBCFitResult:
    dataset = build_state_objective_bc_dataset_from_frame(df, targets=targets, profile=profile, sample_size=int(sample_size), rng_seed=int(rng_seed))
    _policy, actor = build_policy(observation_dim_for_profile(profile), profile)
    weights = state_objective_bc_sample_weights(dataset.metadata, actions=dataset.actions, config=sample_weight_config)
    before = evaluate_bc_actor(actor, dataset.obs, dataset.actions)
    train = _position_aware_supervised_train(
        actor,
        dataset.obs,
        dataset.actions,
        weights,
        steps=int(train_config.steps),
        lr=float(train_config.lr),
        batch_size=int(train_config.batch_size),
        ordinal_lambda=float(position_loss_config.get('ordinal_lambda', 0.0)),
    )
    after = evaluate_bc_actor(actor, dataset.obs, dataset.actions)
    metrics = {
        'samples': int(after['samples']),
        'steps': int(train.get('steps', train_config.steps)),
        'lr': float(train_config.lr),
        'batch_size': int(train_config.batch_size),
        'before_loss': float(before['loss']),
        'before_accuracy': float(before['accuracy']),
        'train_last_loss': float(train['loss']),
        'train_last_ce_loss': float(train['ce_loss']),
        'train_last_ordinal_loss': float(train['ordinal_loss']),
        'train_last_batch_accuracy': float(train['accuracy']),
        'after_loss': float(after['loss']),
        'after_accuracy': float(after['accuracy']),
        'weight_mean': float(weights.mean()),
        'weight_min': float(weights.min()),
        'weight_max': float(weights.max()),
        'ordinal_lambda': float(position_loss_config.get('ordinal_lambda', 0.0)),
    }
    action_count = int(getattr(actor, 'output_dim', 0) or (int(dataset.actions.max()) + 1 if len(dataset.actions) else 0))
    return StateObjectiveBCFitResult(actor=actor, obs=dataset.obs, actions=dataset.actions, metrics=metrics, action_distribution=action_distribution_frame(dataset.actions, action_count=action_count))


def evaluate_action_calibration(actor: Any, dataset: Any, *, slice_cols: tuple[str, ...]) -> pd.DataFrame:
    labels = np.asarray(dataset.actions, dtype=np.int64)
    preds = _actor_predictions_for_notebook(actor, dataset.obs)
    work = dataset.metadata.copy().reset_index(drop=True)
    work['_label_action'] = labels
    work['_predicted_action'] = preds
    work['_action_error'] = preds - labels
    work['_abs_action_error'] = np.abs(work['_action_error'])
    work['_severe_action_error'] = work['_abs_action_error'] >= 2
    rows: list[dict[str, Any]] = []

    def append_row(name: str, mask: pd.Series) -> None:
        subset = work[mask.to_numpy(dtype=bool)]
        samples = int(len(subset))
        if samples == 0:
            rows.append({'slice': name, 'samples': 0, 'action_accuracy': float('nan'), 'action_mae': float('nan'), 'action_rmse': float('nan'), 'severe_action_error_share': float('nan'), 'predicted_action_mean': float('nan'), 'label_action_mean': float('nan'), 'under_action_share': float('nan'), 'over_action_share': float('nan')})
            return
        rows.append({'slice': name, 'samples': samples, 'action_accuracy': float((subset['_label_action'] == subset['_predicted_action']).mean()), 'action_mae': float(subset['_abs_action_error'].mean()), 'action_rmse': float(np.sqrt(np.square(subset['_action_error']).mean())), 'severe_action_error_share': float(subset['_severe_action_error'].mean()), 'predicted_action_mean': float(subset['_predicted_action'].mean()), 'label_action_mean': float(subset['_label_action'].mean()), 'under_action_share': float((subset['_predicted_action'] < subset['_label_action']).mean()), 'over_action_share': float((subset['_predicted_action'] > subset['_label_action']).mean())})

    append_row('all', pd.Series(True, index=work.index))
    for col in slice_cols:
        if col in work.columns:
            append_row(str(col), work[col].fillna(False).astype(bool))
    return pd.DataFrame(rows).sort_values(['slice']).reset_index(drop=True)


In [5]:

source_summary = json.loads(SOURCE_SUMMARY_PATH.read_text(encoding='utf-8'))
profile = load_profile(PROFILE_NAME)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
for col in ['release_gate_state', 'aux_state_head_release_state', 'is_rebound_cluster', 'is_downtrend_state', 'drawdown_control_state', 'failedfold_state_head_target_state', 'is_high_participation_target']:
    if col in targets.columns:
        targets[col] = bool_series(targets, col)
runnable_reference = pd.read_csv(RUNNABLE_REFERENCE_PATH)
soxl_baseline_metrics = pd.read_csv(SOXL_BASELINE_METRICS_PATH)
runnable_by_fold = baseline_metrics_by_fold(runnable_reference, candidate='bull65_hold3')
soxl_default_by_fold = baseline_metrics_by_fold(soxl_baseline_metrics, candidate='legacy_default')
soxl_risk_floor_by_fold = baseline_metrics_by_fold(soxl_baseline_metrics, candidate='wf_pass_ytd18_15_15_02')

init_qlib(QLIB_DATA_PATH)

coverage_rows: list[dict[str, Any]] = []
split_rows: list[pd.DataFrame] = []
fold_cache: dict[str, dict[str, Any]] = {}
for fold, spec in WALK_FORWARD_WINDOWS.items():
    target_fold = str(spec['target_fold'])
    train_start, train_end = map(pd.Timestamp, spec['train'])
    test_start, test_end = map(pd.Timestamp, spec['test'])
    fold_targets_all = targets[targets['fold'].astype(str).eq(target_fold)].copy()
    fold_targets = fold_targets_all[(fold_targets_all['datetime'] >= train_start) & (fold_targets_all['datetime'] <= train_end)].copy()
    split_targets = stratified_target_split(fold_targets)
    split_targets.insert(0, 'wf_window', fold)
    split_rows.append(split_targets)
    fit_targets = split_targets[split_targets['bc_target_split'].eq('train')].drop(columns=['_target_row_number']).copy()
    holdout_targets = split_targets[split_targets['bc_target_split'].eq('holdout')].drop(columns=['_target_row_number']).copy()
    overlap = split_targets[(split_targets['datetime'] >= test_start) & (split_targets['datetime'] <= test_end)]

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = datetime_index_frame(flatten_feature_frame(dataset.handler.feature_frame))
    price_full = datetime_index_series(dataset.handler.price_frame['price'])
    train_frame = feature_full.loc[spec['train'][0]:spec['train'][1]].copy()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec['test'], lookback=LOOKBACK)
    full_counts = slice_count_map(split_targets)
    train_counts = slice_count_map(fit_targets)
    holdout_counts = slice_count_map(holdout_targets)
    fold_cache[fold] = {
        'dataset': dataset,
        'feature_full': feature_full,
        'price_full': price_full,
        'train_frame': train_frame,
        'warm_seg': warm_seg,
        'targets': split_targets,
        'fit_targets': fit_targets,
        'holdout_targets': holdout_targets,
        'full_slice_counts': full_counts,
        'train_slice_counts': train_counts,
        'holdout_slice_counts': holdout_counts,
    }
    row = {
        'fold': fold,
        'target_fold': target_fold,
        'target_rows': int(len(split_targets)),
        'fit_target_rows': int(len(fit_targets)),
        'holdout_target_rows': int(len(holdout_targets)),
        'target_start': str(split_targets['datetime'].min().date()) if len(split_targets) else '',
        'target_end': str(split_targets['datetime'].max().date()) if len(split_targets) else '',
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'feature_start': str(feature_full.index.min().date()),
        'feature_end': str(feature_full.index.max().date()),
        'train_frame_rows': int(len(train_frame)),
        'warm_start': warm_seg[0],
        'warm_end': warm_seg[1],
        'eval_overlap_rows': int(len(overlap)),
        'fold_local_gate': bool(len(split_targets) > 0 and len(overlap) == 0),
    }
    for prefix, counts in [('full', full_counts), ('fit', train_counts), ('holdout', holdout_counts)]:
        for col, value in counts.items():
            row[f'{prefix}_{col}_rows'] = int(value)
    coverage_rows.append(row)

target_split = pd.concat(split_rows, ignore_index=True) if split_rows else pd.DataFrame()
target_split.to_csv(CARD / 'target_holdout_split.csv', index=False)
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'target_coverage.csv', index=False)
print(coverage.to_string(index=False))


[16489:MainThread](2026-06-15 11:31:31,825) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[16489:MainThread](2026-06-15 11:31:32,226) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[16489:MainThread](2026-06-15 11:31:32,227) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[16489:MainThread](2026-06-15 11:31:33,165) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[16489:MainThread](2026-06-15 11:31:33,165) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[16489:MainThread](2026-06-15 11:31:34,198) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[16489:MainThread](2026-06-15 11:31:35,418) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[16489:MainThread](2026-06-15 11:31:35,418) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[16489:MainThread](2026-06-15 11:31:35,419) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


        fold        target_fold  target_rows  fit_target_rows  holdout_target_rows target_start target_end train_start  train_end test_start   test_end feature_start feature_end  train_frame_rows warm_start   warm_end  eval_overlap_rows  fold_local_gate  full_all_rows  full_aux_state_head_release_state_rows  full_is_rebound_cluster_rows  full_is_downtrend_state_rows  full_drawdown_control_state_rows  full_failedfold_state_head_target_state_rows  full_release_gate_state_rows  fit_all_rows  fit_aux_state_head_release_state_rows  fit_is_rebound_cluster_rows  fit_is_downtrend_state_rows  fit_drawdown_control_state_rows  fit_failedfold_state_head_target_state_rows  fit_release_gate_state_rows  holdout_all_rows  holdout_aux_state_head_release_state_rows  holdout_is_rebound_cluster_rows  holdout_is_downtrend_state_rows  holdout_drawdown_control_state_rows  holdout_failedfold_state_head_target_state_rows  holdout_release_gate_state_rows
     wf_2020      wf_2020_train         2470             

In [6]:

candidate_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
calibration_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []
daily_manifest: list[dict[str, Any]] = []
drawdown_rows: list[dict[str, Any]] = []
soxl_baseline_rows: list[dict[str, Any]] = []

for fold, spec in WALK_FORWARD_WINDOWS.items():
    cache = fold_cache[fold]
    fit_targets = cache['fit_targets']
    holdout_targets = cache['holdout_targets']
    if fit_targets.empty or holdout_targets.empty:
        candidate_rows.append({'fold': fold, 'candidate': 'none', 'status': 'skipped_empty_split'})
        continue
    for i, (candidate, weight_config) in enumerate(BC_WEIGHT_CONFIGS.items()):
        print(f'[fold:{fold}] candidate={candidate} fit_targets={len(fit_targets)} holdout={len(holdout_targets)} warm={cache["warm_seg"]}')
        train_config = BC_TRAIN_CONFIGS[candidate]
        set_seed(int(train_config.seed) + i * 101 + list(WALK_FORWARD_WINDOWS).index(fold) * 1009)
        fit = fit_position_aware_bc_actor_from_frame(
            cache['train_frame'],
            targets=fit_targets,
            profile=profile,
            train_config=train_config,
            sample_weight_config=weight_config,
            position_loss_config=POSITION_LOSS_CONFIGS[candidate],
            sample_size=0,
            rng_seed=int(train_config.seed),
        )
        fit_row = {
            'fold': fold,
            'candidate': candidate,
            'fit_target_rows': int(len(fit_targets)),
            'holdout_target_rows': int(len(holdout_targets)),
            'bc_samples': int(len(fit.actions)),
            'unique_actions': int(pd.Series(fit.actions).nunique()) if len(fit.actions) else 0,
            'train_config': json.dumps(asdict(train_config), sort_keys=True),
            'weight_config': json.dumps(asdict(weight_config), sort_keys=True),
            'position_loss_config': json.dumps(POSITION_LOSS_CONFIGS[candidate], sort_keys=True),
        }
        fit_row.update({k: float(v) if isinstance(v, (int, float, np.number)) and math.isfinite(float(v)) else v for k, v in fit.metrics.items()})
        fit_rows.append(fit_row)

        train_dataset = build_state_objective_bc_dataset_from_frame(cache['train_frame'], targets=fit_targets, profile=profile, sample_size=0)
        holdout_dataset = build_state_objective_bc_dataset_from_frame(cache['train_frame'], targets=holdout_targets, profile=profile, sample_size=0)
        train_slice_metrics = evaluate_bc_actor_slice_fit(fit.actor, fit.obs, fit.actions, train_dataset.metadata, slice_cols=SLICE_COLS)
        holdout_slice_metrics = evaluate_bc_actor_slice_fit(fit.actor, holdout_dataset.obs, holdout_dataset.actions, holdout_dataset.metadata, slice_cols=SLICE_COLS)
        train_slice_gate, train_slice_details, train_failed_slices = slice_gate_from_metrics(
            train_slice_metrics,
            fold=fold,
            full_target_counts=cache['train_slice_counts'],
        )
        holdout_slice_gate, holdout_slice_details, holdout_failed_slices = slice_gate_from_metrics(
            holdout_slice_metrics,
            fold=fold,
            full_target_counts=cache['full_slice_counts'],
        )
        for split_name, metrics_frame in [('train', train_slice_metrics), ('holdout', holdout_slice_metrics)]:
            tagged = metrics_frame.copy()
            tagged.insert(0, 'bc_split', split_name)
            tagged.insert(0, 'candidate', candidate)
            tagged.insert(0, 'fold', fold)
            slice_frames.append(tagged)

        train_calibration = evaluate_action_calibration(fit.actor, train_dataset, slice_cols=SLICE_COLS)
        holdout_calibration = evaluate_action_calibration(fit.actor, holdout_dataset, slice_cols=SLICE_COLS)
        for split_name, calibration_frame in [('train', train_calibration), ('holdout', holdout_calibration)]:
            tagged = calibration_frame.copy()
            tagged.insert(0, 'bc_split', split_name)
            tagged.insert(0, 'candidate', candidate)
            tagged.insert(0, 'fold', fold)
            calibration_frames.append(tagged)
        holdout_calibration_index = holdout_calibration.set_index('slice') if not holdout_calibration.empty else pd.DataFrame()
        holdout_all_calibration = holdout_calibration_index.loc['all'].to_dict() if 'all' in holdout_calibration_index.index else {}

        action_dist = fit.action_distribution.copy()
        action_dist.insert(0, 'candidate', candidate)
        action_dist.insert(0, 'fold', fold)
        action_frames.append(action_dist)

        bt = backtest_profile(
            fit.actor,
            cache['feature_full'],
            cache['price_full'],
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            mode_router_config=profile.mode_router,
            asset_idx=0,
            feature_columns=feature_columns,
            model_config=profile.model,
            test_seg=cache['warm_seg'],
            trace_decisions=True,
        )
        daily = slice_warm_start_daily_path(
            portfolio_df=bt['portfolio'],
            price_series=cache['price_full'],
            eval_seg=spec['test'],
            position_history=bt.get('position_history', []),
            trade_signals=bt.get('trade_signals', []),
        )
        daily_path = DAILY_DIR / f'{fold}__{candidate}_official_daily_path.csv'
        daily.to_csv(daily_path, index=False)
        metrics = daily_path_metrics(daily)
        drawdown = max_drawdown_interval_metrics(daily)
        drawdown_rows.append({'fold': fold, 'candidate': candidate, **drawdown, 'daily_path': str(daily_path.relative_to(ROOT))})
        daily_manifest.append({
            'fold': fold,
            'candidate': candidate,
            'path': str(daily_path.relative_to(ROOT)),
            'rows': int(len(daily)),
            'start': metrics.get('start'),
            'end': metrics.get('end'),
        })

        runnable_base = runnable_by_fold.get(fold, {})
        drift_score = score_bc_walkforward_drift(
            fold,
            candidate_metrics=metrics,
            baseline_metrics=runnable_base,
            slice_gate=holdout_slice_gate,
            target_relative_return_pp=TARGET_RELATIVE_RETURN,
            target_max_dd_pct=TARGET_MAX_DD,
            relative_drift_floor_pp=DRIFT_RELATIVE_FLOOR_PP,
            max_dd_drift_ceiling_pp=DRIFT_MAX_DD_CEILING_PP,
        )
        strict_max_dd_gate = bool(float(metrics['max_dd']) < TARGET_MAX_DD)
        relative_gate = bool(float(metrics['relative_return']) >= TARGET_RELATIVE_RETURN)
        candidate_gate_strict = bool(relative_gate and strict_max_dd_gate and holdout_slice_gate and bool(drift_score['drift_gate']))
        holdout_index = holdout_slice_metrics.set_index('slice') if not holdout_slice_metrics.empty else pd.DataFrame()
        required_holdout_details = [
            value for value in holdout_slice_details.values()
            if bool(value.get('required')) and int(value.get('samples') or 0) > 0
        ]
        predicted_unique_min = min(
            (int(value.get('predicted_unique_actions') or 0) for value in required_holdout_details),
            default=0,
        )
        row = {
            'fold': fold,
            'candidate': candidate,
            'status': 'evaluated',
            'warm_start': cache['warm_seg'][0],
            'warm_end': cache['warm_seg'][1],
            'fit_target_rows': int(len(fit_targets)),
            'holdout_target_rows': int(len(holdout_targets)),
            'bc_samples': int(len(fit.actions)),
            'holdout_samples': int(len(holdout_dataset.actions)),
            'return': float(metrics['return']),
            'buy_and_hold_return': float(metrics['buy_and_hold_return']),
            'relative_return': float(metrics['relative_return']),
            'max_dd': float(metrics['max_dd']),
            'avg_pos': float(metrics['avg_pos']),
            'trade_count': int(pd.Series(daily['action']).astype(str).str.upper().isin(['BUY', 'SELL']).sum()),
            'target_relative_return': TARGET_RELATIVE_RETURN,
            'target_max_dd_strict_lt': TARGET_MAX_DD,
            'relative_gate': relative_gate,
            'max_dd_gate_strict': strict_max_dd_gate,
            'train_slice_gate': bool(train_slice_gate),
            'holdout_slice_gate': bool(holdout_slice_gate),
            'train_failed_slices': ','.join(train_failed_slices),
            'holdout_failed_slices': ','.join(holdout_failed_slices),
            'holdout_required_predicted_unique_actions_min': predicted_unique_min,
            'holdout_action_mae_all': float(holdout_all_calibration.get('action_mae', float('nan'))),
            'holdout_severe_action_error_share_all': float(holdout_all_calibration.get('severe_action_error_share', float('nan'))),
            'holdout_under_action_share_all': float(holdout_all_calibration.get('under_action_share', float('nan'))),
            'holdout_over_action_share_all': float(holdout_all_calibration.get('over_action_share', float('nan'))),
            'drift_gate': bool(drift_score['drift_gate']),
            'candidate_gate_strict': candidate_gate_strict,
            'relative_delta_vs_runnable': float(drift_score['relative_delta_vs_baseline']),
            'max_dd_delta_vs_runnable': float(drift_score['max_dd_delta_vs_baseline']),
            'runnable_reference_relative': float(drift_score['baseline_relative_return']),
            'runnable_reference_max_dd': float(drift_score['baseline_max_dd']),
            'runnable_reference_avg_pos': float(drift_score['baseline_avg_pos']),
            'daily_path': str(daily_path.relative_to(ROOT)),
            'train_slice_details_json': json.dumps(json_safe(train_slice_details), sort_keys=True),
            'holdout_slice_details_json': json.dumps(json_safe(holdout_slice_details), sort_keys=True),
        }
        row['selection_loss'] = candidate_loss(row)
        candidate_rows.append(row)

        for label, baseline in [('runnable_reference', runnable_base), ('soxl_legacy_default', soxl_default_by_fold.get(fold, {})), ('soxl_risk_floor', soxl_risk_floor_by_fold.get(fold, {}))]:
            soxl_baseline_rows.append({
                'fold': fold,
                'candidate': candidate,
                'baseline': label,
                'candidate_relative_return': float(metrics['relative_return']),
                'candidate_max_dd': float(metrics['max_dd']),
                'candidate_avg_pos': float(metrics['avg_pos']),
                'baseline_relative_return': float(baseline.get('relative_return', float('nan'))),
                'baseline_max_dd': float(baseline.get('max_dd', float('nan'))),
                'baseline_avg_pos': float(baseline.get('avg_pos', float('nan'))),
                'relative_delta': float(metrics['relative_return']) - float(baseline.get('relative_return', float('nan'))),
                'max_dd_delta': float(metrics['max_dd']) - float(baseline.get('max_dd', float('nan'))),
            })

candidate_metrics = pd.DataFrame(candidate_rows).sort_values(['fold', 'selection_loss', 'candidate']).reset_index(drop=True)
fit_metrics = pd.DataFrame(fit_rows).sort_values(['fold', 'candidate']).reset_index(drop=True)
slice_metrics_all = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()
action_calibration = pd.concat(calibration_frames, ignore_index=True) if calibration_frames else pd.DataFrame()
daily_manifest_df = pd.DataFrame(daily_manifest)
drawdown_df = pd.DataFrame(drawdown_rows)
soxl_baseline_comparison = pd.DataFrame(soxl_baseline_rows)

candidate_metrics.to_csv(CARD / 'bc_candidate_metrics.csv', index=False)
fit_metrics.to_csv(CARD / 'bc_fit_metrics.csv', index=False)
slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
action_distribution.to_csv(CARD / 'bc_action_distribution.csv', index=False)
action_calibration.to_csv(CARD / 'bc_action_calibration.csv', index=False)
daily_manifest_df.to_csv(CARD / 'official_daily_path_manifest.csv', index=False)
drawdown_df.to_csv(CARD / 'drawdown_interval_diagnostic.csv', index=False)
soxl_baseline_comparison.to_csv(CARD / 'soxlbacktest_baseline_comparison.csv', index=False)
print(candidate_metrics[['fold','candidate','relative_return','max_dd','holdout_slice_gate','holdout_failed_slices','holdout_action_mae_all','holdout_severe_action_error_share_all','drift_gate','selection_loss']].to_string(index=False))


[fold:wf_2020] candidate=ordinal_low_aux fit_targets=1975 holdout=495 warm=('2019-10-07', '2020-12-31')


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[16489:MainThread](2026-06-15 11:31:40,748) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!


[16489:MainThread](2026-06-15 11:31:40,749) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


[16489:MainThread](2026-06-15 11:31:40,751) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x138c34b10>


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


        fold            candidate  relative_return    max_dd  holdout_slice_gate                         holdout_failed_slices  holdout_action_mae_all  holdout_severe_action_error_share_all  drift_gate  selection_loss
     wf_2020      ordinal_mid_aux       -13.659859 25.580808                True                                                              1.361111                               0.446759       False      108.330375
     wf_2020      ordinal_low_aux       -18.912679 25.314497               False all,is_rebound_cluster,drawdown_control_state                1.412037                               0.476852       False      142.946482
     wf_2020 ordinal_drawdown_aux       -28.272235 33.111694                True                                                              1.356481                               0.437500       False      182.610813
     wf_2020     ordinal_high_aux       -62.904341 34.207435               False                            is_rebound_cluster  

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

In [7]:

selection_rows: list[dict[str, Any]] = []
for fold in WALK_FORWARD_WINDOWS:
    fold_candidates = candidate_metrics[candidate_metrics['fold'].astype(str).eq(fold)].copy()
    if fold_candidates.empty:
        selection_rows.append({'fold': fold, 'status': 'missing', 'selected_candidate': None})
        continue
    selected = fold_candidates.sort_values(
        ['candidate_gate_strict', 'holdout_slice_gate', 'selection_loss', 'candidate'],
        ascending=[False, False, True, True],
    ).iloc[0].to_dict()
    selection_rows.append({
        'fold': fold,
        'status': 'evaluated',
        'selected_candidate': selected['candidate'],
        'selected_candidate_gate': bool(selected['candidate_gate_strict']),
        'selected_relative_return': float(selected['relative_return']),
        'selected_max_dd': float(selected['max_dd']),
        'selected_avg_pos': float(selected['avg_pos']),
        'selected_trade_count': int(selected['trade_count']),
        'selected_loss': float(selected['selection_loss']),
        'relative_gate': bool(selected['relative_gate']),
        'max_dd_gate_strict': bool(selected['max_dd_gate_strict']),
        'train_slice_gate': bool(selected['train_slice_gate']),
        'holdout_slice_gate': bool(selected['holdout_slice_gate']),
        'drift_gate': bool(selected['drift_gate']),
        'train_failed_slices': selected.get('train_failed_slices', ''),
        'holdout_failed_slices': selected.get('holdout_failed_slices', ''),
        'relative_delta_vs_runnable': float(selected['relative_delta_vs_runnable']),
        'max_dd_delta_vs_runnable': float(selected['max_dd_delta_vs_runnable']),
        'daily_path': selected['daily_path'],
    })

selection_by_fold = pd.DataFrame(selection_rows)
selection_by_fold.to_csv(CARD / 'bc_selection_by_fold.csv', index=False)

# Re-score selected rows through the shared drift summarizer for compact gate accounting.
drift_scores = []
for _, row in selection_by_fold.iterrows():
    fold = str(row['fold'])
    drift_scores.append(score_bc_walkforward_drift(
        fold,
        candidate_metrics={
            'relative_return': row.get('selected_relative_return'),
            'max_dd': row.get('selected_max_dd'),
            'avg_pos': row.get('selected_avg_pos'),
        },
        baseline_metrics=runnable_by_fold.get(fold, {}),
        slice_gate=bool(row.get('holdout_slice_gate')),
        target_relative_return_pp=TARGET_RELATIVE_RETURN,
        target_max_dd_pct=TARGET_MAX_DD,
        relative_drift_floor_pp=DRIFT_RELATIVE_FLOOR_PP,
        max_dd_drift_ceiling_pp=DRIFT_MAX_DD_CEILING_PP,
    ))
drift_scores_df = pd.DataFrame(drift_scores)
drift_scores_df.to_csv(CARD / 'bc_drift_scores.csv', index=False)
drift_summary = summarize_bc_walkforward_drift_screen(drift_scores_df, required_folds=tuple(WALK_FORWARD_WINDOWS))

required = list(WALK_FORWARD_WINDOWS)
selected_required = selection_by_fold[selection_by_fold['fold'].isin(required)].copy()
selected_gate_pass_count = int(selected_required['selected_candidate_gate'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
holdout_slice_gate_pass_count = int(selected_required['holdout_slice_gate'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
train_slice_gate_pass_count = int(selected_required['train_slice_gate'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
drift_gate_pass_count = int(selected_required['drift_gate'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
candidate_relative_pass_count = int(selected_required['relative_gate'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
strict_dd_pass_count = int(selected_required['max_dd_gate_strict'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
failed_folds = selected_required.loc[~selected_required['selected_candidate_gate'].fillna(False).astype(bool), 'fold'].astype(str).tolist()
missing_folds = [fold for fold in required if fold not in set(selected_required['fold'].astype(str))]
failed_folds.extend(missing_folds)

if selected_gate_pass_count == len(required):
    decision = 'release16_action_head_position_loss_replay_passed_no_actor_retention'
    next_mechanism = 'run_full_fixed_split_and_retention_review_before_actor_retention'
elif holdout_slice_gate_pass_count < len(required):
    if holdout_slice_gate_pass_count > 0:
        decision = 'release16_action_head_position_loss_partial_repair_downtrend_failed_no_promotion'
    else:
        decision = 'release16_action_head_position_loss_slice_failed_no_promotion'
    next_mechanism = 'add_explicit_downtrend_drawdown_aux_state_head_or_checkpoint_objective_before_official_replay'
elif strict_dd_pass_count < len(required):
    decision = 'release16_action_head_position_loss_drawdown_failed_no_promotion'
    next_mechanism = 'add_drawdown_drift_checkpoint_objective_before_ppo_or_retention_training'
elif candidate_relative_pass_count < len(required):
    decision = 'release16_action_head_position_loss_relative_floor_failed_no_promotion'
    next_mechanism = 'revise_participation_target_or_checkpoint_objective_before_ppo'
elif drift_gate_pass_count < len(required):
    decision = 'release16_action_head_position_loss_drift_failed_no_promotion'
    next_mechanism = 'use_bc_as_reference_and_add_drift_stop_before_longer_ppo'
else:
    decision = 'release16_action_head_position_loss_mixed_no_promotion'
    next_mechanism = 'inspect_failed_fold_action_distribution_then_try_drawdown_drift_reward_or_checkpoint_selector'

selected_by_fold: dict[str, dict[str, Any]] = {}
metrics: list[dict[str, Any]] = []
for _, row in selected_required.iterrows():
    fold = str(row['fold'])
    selected_by_fold[fold] = {
        'selected_candidate': str(row['selected_candidate']),
        'selected_candidate_gate': bool(row['selected_candidate_gate']),
        'selected_relative_return': float(row['selected_relative_return']),
        'selected_max_dd': float(row['selected_max_dd']),
        'selected_avg_pos': float(row['selected_avg_pos']),
        'selected_trade_count': int(row['selected_trade_count']),
        'relative_gate': bool(row['relative_gate']),
        'max_dd_gate_strict': bool(row['max_dd_gate_strict']),
        'train_slice_gate': bool(row['train_slice_gate']),
        'holdout_slice_gate': bool(row['holdout_slice_gate']),
        'drift_gate': bool(row['drift_gate']),
        'train_failed_slices': str(row.get('train_failed_slices', '')),
        'holdout_failed_slices': str(row.get('holdout_failed_slices', '')),
        'relative_delta_vs_runnable': float(row['relative_delta_vs_runnable']),
        'max_dd_delta_vs_runnable': float(row['max_dd_delta_vs_runnable']),
    }
    metrics.append({
        'window': fold,
        'split_type': 'walk_forward_test',
        'relative_return_pct': float(row['selected_relative_return']),
        'max_dd_pct': float(row['selected_max_dd']),
        'avg_pos': float(row['selected_avg_pos']),
    })

min_relative = min((float(r['relative_return_pct']) for r in metrics), default=float('nan'))
worst_max_dd = max((float(r['max_dd_pct']) for r in metrics), default=float('nan'))
primary_blocker = 'downtrend_drawdown_holdout_slice_failed' if holdout_slice_gate_pass_count < len(required) else 'joint_drawdown_return_gap'
if selected_gate_pass_count == len(required):
    primary_blocker = 'diagnostic_no_promotion'
summary = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'bc_warm_start', 'bc_position_loss', 'official_replay'],
    'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
    'direct_comparison_baseline': 'release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1',
    'source_card': 'release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1',
    'source_target_contract': 'release16_joint_drawdown_release_target_contract_v1/joint_target_contract.csv',
    'source_paths': [
        str((SOURCE_SUMMARY_PATH).relative_to(ROOT)),
        str((TARGETS_PATH).relative_to(ROOT)),
        str((RUNNABLE_REFERENCE_PATH).relative_to(ROOT)),
        str((SOXL_BASELINE_METRICS_PATH).relative_to(ROOT)),
    ],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'ppo_allowed_next': False,
    'bc_screen_allowed_next': False,
    'source_target_contract_gate': bool(source_summary.get('target_contract_gate', False)),
    'source_bc_readiness_gate': bool(source_summary.get('bc_readiness_gate', False)),
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'drawdown_gate_operator': '<',
    'target_window': required,
    'metrics': metrics,
    'selected_by_fold': selected_by_fold,
    'failed_folds': failed_folds,
    'selected_gate_pass_count': selected_gate_pass_count,
    'candidate_relative_pass_count': candidate_relative_pass_count,
    'strict_dd_pass_count': strict_dd_pass_count,
    'train_slice_gate_pass_count': train_slice_gate_pass_count,
    'holdout_slice_gate_pass_count': holdout_slice_gate_pass_count,
    'drift_gate_pass_count': drift_gate_pass_count,
    'min_wf_relative_return_pp': min_relative,
    'worst_wf_max_drawdown_pct': worst_max_dd,
    'max_relative_gap_to_target_pp': max(0.0, TARGET_RELATIVE_RETURN - min_relative) if math.isfinite(min_relative) else None,
    'max_drawdown_gap_to_target_pp': max(0.0, worst_max_dd - TARGET_MAX_DD) if math.isfinite(worst_max_dd) else None,
    'drift_summary': drift_summary,
    'next_mechanism': next_mechanism,
    'primary_blocker': primary_blocker,
    'training_allowed_from_this_card': False,
    'summary_points': [
        f'Selected BC candidates passed {selected_gate_pass_count}/{len(required)} full fold gates under official warm-start replay.',
        f'Holdout slice gate passed {holdout_slice_gate_pass_count}/{len(required)} folds; train slice gate passed {train_slice_gate_pass_count}/{len(required)} folds.',
        f'Drift gate passed {drift_gate_pass_count}/{len(required)} folds.',
        f'Min selected relative return is {min_relative:.2f}pp and worst selected MaxDD is {worst_max_dd:.2f}%.',
        'Holdout action-calibration metrics are written as bc_action_calibration.csv.',
        'SOXLBacktest qlib baseline comparison is written as soxlbacktest_baseline_comparison.csv.',
        'No actor is retained from this diagnostic screen.',
    ],
    'sidecars': {
        'target_coverage': str((CARD / 'target_coverage.csv').relative_to(ROOT)),
        'target_holdout_split': str((CARD / 'target_holdout_split.csv').relative_to(ROOT)),
        'bc_fit_metrics': str((CARD / 'bc_fit_metrics.csv').relative_to(ROOT)),
        'bc_candidate_metrics': str((CARD / 'bc_candidate_metrics.csv').relative_to(ROOT)),
        'bc_selection_by_fold': str((CARD / 'bc_selection_by_fold.csv').relative_to(ROOT)),
        'bc_slice_metrics': str((CARD / 'bc_slice_metrics.csv').relative_to(ROOT)),
        'bc_action_distribution': str((CARD / 'bc_action_distribution.csv').relative_to(ROOT)),
        'bc_action_calibration': str((CARD / 'bc_action_calibration.csv').relative_to(ROOT)),
        'bc_drift_scores': str((CARD / 'bc_drift_scores.csv').relative_to(ROOT)),
        'drawdown_interval_diagnostic': str((CARD / 'drawdown_interval_diagnostic.csv').relative_to(ROOT)),
        'official_daily_path_manifest': str((CARD / 'official_daily_path_manifest.csv').relative_to(ROOT)),
        'soxlbacktest_baseline_comparison': str((CARD / 'soxlbacktest_baseline_comparison.csv').relative_to(ROOT)),
        'selection': str((CARD / 'selection.json').relative_to(ROOT)),
        'summary': str((CARD / 'summary.json').relative_to(ROOT)),
        'conclusion': str((CARD / 'conclusion.md').relative_to(ROOT)),
    },
}
selection = {
    'experiment': EXPERIMENT,
    'created_at': summary['created_at'],
    'decision': decision,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'source_card': 'release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1',
    'source_target_contract': 'release16_joint_drawdown_release_target_contract_v1/joint_target_contract.csv',
    'selected_by_fold': selected_by_fold,
    'failed_folds': failed_folds,
    'primary_blocker': primary_blocker,
    'next_mechanism': next_mechanism,
    'sidecars': summary['sidecars'],
}
write_json(CARD / 'summary.json', summary)
write_json(CARD / 'selection.json', selection)

conclusion_lines = [
    f'# {EXPERIMENT}',
    '',
    f'Decision: `{decision}`.',
    '',
    f'Full fold gates passed: `{selected_gate_pass_count}/{len(required)}`.',
    f'Holdout slice gates passed: `{holdout_slice_gate_pass_count}/{len(required)}`.',
    f'Strict drawdown gates passed: `{strict_dd_pass_count}/{len(required)}`.',
    f'Relative return gates passed: `{candidate_relative_pass_count}/{len(required)}`.',
    f'Drift gates passed: `{drift_gate_pass_count}/{len(required)}`.',
    '',
    f'Min selected relative return: `{min_relative:.2f}pp`; worst selected MaxDD: `{worst_max_dd:.2f}%`.',
    '',
    'Selected by fold:',
]
for fold, payload in selected_by_fold.items():
    conclusion_lines.append(
        f'- `{fold}`: `{payload["selected_candidate"]}`, rel `{payload["selected_relative_return"]:.2f}pp`, MaxDD `{payload["selected_max_dd"]:.2f}%`, holdout_slice_gate `{payload["holdout_slice_gate"]}`, failed `{payload["holdout_failed_slices"]}`.'
    )
conclusion_lines.extend([
    '',
    'Risk note: this is a diagnostic BC holdout and official replay screen. It does not retain an actor and does not authorize PPO or production promotion.',
    '',
    f'Next action: `{next_mechanism}`.',
    '',
])
(CARD / 'conclusion.md').write_text('\n'.join(conclusion_lines), encoding='utf-8')
print(json.dumps(json_safe(summary), indent=2, sort_keys=True))


{
  "actor_retention_allowed": false,
  "bc_screen_allowed_next": false,
  "candidate_relative_pass_count": 2,
  "changed_layer": [
    "model_selection",
    "bc_warm_start",
    "bc_position_loss",
    "official_replay"
  ],
  "created_at": "2026-06-15T03:32:31.545361+00:00",
  "decision": "release16_action_head_position_loss_partial_repair_downtrend_failed_no_promotion",
  "direct_comparison_baseline": "release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1",
  "drawdown_gate_operator": "<",
  "drift_gate_pass_count": 1,
  "drift_summary": {
    "baseline_gate_pass_count": 1,
    "candidate_gate_pass_count": 1,
    "decision": "bc_walkforward_drift_failed_no_promotion",
    "drift_gate_pass_count": 1,
    "failed_folds": [
      "wf_2020",
      "wf_2022",
      "wf_2023_2024"
    ],
    "fold_gate_pass_count": 0,
    "next_mechanism": "block_ppo_until_drawdown_drift_or_slice_fit_is_fixed",
    "promotion_allowed": false,
    "required_folds": [
      "wf_2020",
      "wf_2022

## Conclusion

This notebook writes its executable conclusion to `summary.json`, `selection.json`, and `conclusion.md`. The run is a diagnostic BC holdout and official replay screen only; no actor checkpoint is retained from this card.
